# Human Swiss-Prot ESM-1b embeddings

This notebook builds a complete ESM-1b cache for human Swiss-Prot proteins. It uses `plm.py` as the authoritative embedding implementation and preserves its mean-pooling, batching, and long-sequence behavior.

The existing Swiss-Prot PLM cache is reused only when the accession and amino-acid sequence are identical. Remaining proteins are embedded on the GPU, and the final cache is validated before it is written.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json
import re
import sys

import numpy as np
import pandas as pd
import torch
import transformers

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'plm.py').exists():
    raise FileNotFoundError(f'Run this notebook from the S2F repository or notebooks directory: {PROJECT_ROOT}')
sys.path.insert(0, str(PROJECT_ROOT))

import plm

FULL_SWISSPROT = Path('/media/marcelo_baez/HD_Disc1/databases/ProjectData/swiss-prot/uniprot_sprot.fasta')
FILTERED_SWISSPROT = Path('/media/marcelo_baez/HD_Disc1/.S2F/data/UniprotKB/filtered_sprot')
SOURCE_CACHE = Path('/media/marcelo_baez/HD_Disc1/.S2F/data/PLM/embeddings/target_57e9174250d35f0d')
OUTPUT_CACHE = Path('/media/marcelo_baez/HD_Disc1/.S2F/data/PLM/embeddings/human_9606_all_swissprot')
MODEL_ROOT = PROJECT_ROOT / '.cache' / 'huggingface' / 'models--facebook--esm1b_t33_650M_UR50S' / 'snapshots'
MODEL_PATH = next((p for p in sorted(MODEL_ROOT.glob('*')) if (p / 'pytorch_model.bin').exists()), None)
MODEL_NAME = 'facebook/esm1b_t33_650M_UR50S'
LONG_SEQUENCE_MODE = 'sliding_mean'
LONG_WINDOW_SIZE = 1022
LONG_OVERLAP = 128
BATCH_TOKENS = 2048

for required in [FULL_SWISSPROT, FILTERED_SWISSPROT, SOURCE_CACHE]:
    if not required.exists():
        raise FileNotFoundError(required)
if MODEL_PATH is None:
    raise FileNotFoundError(f'Local Hugging Face model snapshot not found below {MODEL_ROOT}')

print('Python:', sys.executable)
print('PyTorch:', torch.__version__, 'compiled CUDA:', torch.version.cuda)
print('Transformers:', transformers.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('This notebook requires a CUDA-capable kernel before embedding missing proteins.')
print('GPU:', torch.cuda.get_device_name(0))
print('Local ESM-1b model:', MODEL_PATH)
print('Output cache:', OUTPUT_CACHE)

Python: /home/marcelo_baez/paccanaro-lab/PANDA2/.conda-panda2/bin/python
PyTorch: 1.11.0+cu113 compiled CUDA: 11.3
Transformers: 4.46.3
CUDA available: True
GPU: Quadro RTX 5000
Local ESM-1b model: /home/marcelo_baez/paccanaro-lab/S2F/.cache/huggingface/models--facebook--esm1b_t33_650M_UR50S/snapshots/7b37824baec4d3658e1df7479222a7c79b465b76
Output cache: /media/marcelo_baez/HD_Disc1/.S2F/data/PLM/embeddings/human_9606_all_swissprot


## Load and verify human sequences

UniProt accessions come from FASTA headers and human records are selected by the exact taxonomy token `OX=9606`. Duplicate accessions, empty sequences, or mismatched source-cache sequences are rejected.

In [2]:
def is_human(record):
    return re.search(r'(?:^|\s)OX=9606(?:\s|$)', record.header) is not None


all_swissprot_records = plm.read_fasta(FULL_SWISSPROT, protein_id_mode='uniprot')
human_records = [record for record in all_swissprot_records if is_human(record)]
human_by_id = {record.protein_id: record for record in human_records}
if len(human_by_id) != len(human_records):
    raise ValueError('Duplicate human UniProt accessions were found in Swiss-Prot.')
if any(not record.sequence for record in human_records):
    raise ValueError('At least one human Swiss-Prot sequence is empty.')

filtered_records = plm.read_fasta(FILTERED_SWISSPROT, protein_id_mode='uniprot')
filtered_human_by_id = {
    record.protein_id: record
    for record in filtered_records
    if record.protein_id in human_by_id
}

source_cache = plm.load_embedding_cache(SOURCE_CACHE, validate=False)
if source_cache is None:
    raise RuntimeError(f'Incomplete source cache: {SOURCE_CACHE}')
source_index = {protein_id: index for index, protein_id in enumerate(source_cache.ids)}

reusable_ids = []
for protein_id, record in human_by_id.items():
    cached_record = filtered_human_by_id.get(protein_id)
    if protein_id in source_index and cached_record is not None and cached_record.sequence == record.sequence:
        reusable_ids.append(protein_id)

reusable_id_set = set(reusable_ids)
missing_records = [record for record in human_records if record.protein_id not in reusable_id_set]

sequence_summary = pd.DataFrame([
    {
        'human_swissprot_proteins': len(human_records),
        'reused_embeddings': len(reusable_ids),
        'embeddings_to_compute': len(missing_records),
        'sequences_over_1022': sum(len(record.sequence) > LONG_WINDOW_SIZE for record in human_records),
        'missing_sequences_over_1022': sum(len(record.sequence) > LONG_WINDOW_SIZE for record in missing_records),
        'total_residues': sum(len(record.sequence) for record in human_records),
    }
])
display(sequence_summary)

assert len(human_records) == 20420, f'Expected 20,420 human proteins, found {len(human_records):,}'
assert len(reusable_ids) == 16984, f'Expected 16,984 reusable embeddings, found {len(reusable_ids):,}'
assert len(missing_records) == 3436, f'Expected 3,436 missing embeddings, found {len(missing_records):,}'

 human_swissprot_proteins  reused_embeddings  embeddings_to_compute  sequences_over_1022  missing_sequences_over_1022  total_residues
                    20420              16984                   3436                 2302                          267        11413551


## Compute missing embeddings and assemble the complete cache

If a complete validated output cache already exists, it is reused. Otherwise `plm.compute_embeddings` loads ESM-1b once, embeds only missing sequences, and applies sliding-window averaging to long proteins.

In [3]:
expected_metadata = plm.build_cache_metadata(
    [FULL_SWISSPROT],
    model_name=MODEL_NAME,
    protein_id_mode='uniprot',
    long_sequence_mode=LONG_SEQUENCE_MODE,
    long_window_size=LONG_WINDOW_SIZE,
    long_overlap=LONG_OVERLAP,
)

complete_cache = plm.load_embedding_cache(
    OUTPUT_CACHE,
    expected_metadata=expected_metadata,
    validate=True,
)

if complete_cache is not None and complete_cache.ids == [record.protein_id for record in human_records]:
    print(f'Reusing complete human cache: {OUTPUT_CACHE}')
else:
    print(f'Computing {len(missing_records):,} missing human embeddings with plm.py...')
    missing_embeddings = plm.compute_embeddings(
        missing_records,
        model_name=str(MODEL_PATH),
        model_dir=None,
        device_arg='cuda',
        local_files_only=True,
        long_sequence_mode=LONG_SEQUENCE_MODE,
        long_window_size=LONG_WINDOW_SIZE,
        long_overlap=LONG_OVERLAP,
        batch_tokens=BATCH_TOKENS,
    )
    if missing_embeddings.shape != (len(missing_records), 1280):
        raise ValueError(f'Unexpected missing embedding shape: {missing_embeddings.shape}')

    missing_index = {record.protein_id: index for index, record in enumerate(missing_records)}
    complete_embeddings = np.empty((len(human_records), 1280), dtype=np.float32)
    for output_index, record in enumerate(human_records):
        source_position = source_index.get(record.protein_id)
        if record.protein_id in reusable_id_set and source_position is not None:
            complete_embeddings[output_index] = source_cache.embeddings[source_position]
        else:
            complete_embeddings[output_index] = missing_embeddings[missing_index[record.protein_id]]

    metadata = dict(expected_metadata)
    metadata.update({
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'reused_embedding_count': len(reusable_ids),
        'computed_embedding_count': len(missing_records),
        'reused_from': str(SOURCE_CACHE),
        'implementation': str((PROJECT_ROOT / 'plm.py').resolve()),
        'batch_tokens': BATCH_TOKENS,
    })
    complete_cache = plm.write_embedding_cache(
        OUTPUT_CACHE,
        human_records,
        complete_embeddings,
        metadata,
    )
    del missing_embeddings, complete_embeddings
    print(f'Wrote complete human cache: {OUTPUT_CACHE}')

Reusing complete human cache: /media/marcelo_baez/HD_Disc1/.S2F/data/PLM/embeddings/human_9606_all_swissprot


## Final validation

In [4]:
final_embeddings = np.asarray(complete_cache.embeddings)
validation = {
    'cache_path': str(OUTPUT_CACHE.resolve()),
    'protein_count': len(complete_cache.ids),
    'unique_protein_count': len(set(complete_cache.ids)),
    'embedding_shape': list(final_embeddings.shape),
    'dtype': str(final_embeddings.dtype),
    'all_finite': bool(np.isfinite(final_embeddings).all()),
    'zero_vectors': int((np.linalg.norm(final_embeddings, axis=1) == 0).sum()),
    'size_bytes': (OUTPUT_CACHE / 'embeddings.npy').stat().st_size,
    'model_name': complete_cache.metadata.get('model_name'),
    'long_sequence_mode': complete_cache.metadata.get('long_sequence_mode'),
}
assert validation['protein_count'] == 20420
assert validation['unique_protein_count'] == 20420
assert validation['embedding_shape'] == [20420, 1280]
assert validation['all_finite']
assert validation['zero_vectors'] == 0

with (OUTPUT_CACHE / 'validation.json').open('w', encoding='utf-8') as handle:
    json.dump(validation, handle, indent=2, sort_keys=True)

display(pd.DataFrame([validation]))
print('Human ESM cache is ready:', OUTPUT_CACHE.resolve())

                                                                    cache_path  protein_count  unique_protein_count embedding_shape   dtype  all_finite  zero_vectors  size_bytes                    model_name long_sequence_mode
/media/marcelo_baez/HD_Disc1/.S2F/data/PLM/embeddings/human_9606_all_swissprot          20420                 20420   [20420, 1280] float32        True             0   104550528 facebook/esm1b_t33_650M_UR50S       sliding_mean
Human ESM cache is ready: /media/marcelo_baez/HD_Disc1/.S2F/data/PLM/embeddings/human_9606_all_swissprot
